# 01 — Exploration & Préparation du Dataset
## Helpdesk IA — GEISER | PFE 2026

### Objectif
Explorer et préparer le dataset de tickets réels GEISER pour :
- Comprendre la structure et la qualité des données
- Nettoyer le texte (HTML, caractères spéciaux)
- Générer automatiquement les catégories et priorités
- Exporter un dataset propre pour l'entraînement ML

### Dataset brut
- **Source** : Export CSV de helpdesk Geiser
- **Taille** : 1915 tickets
- **Période** : Tickets réels GEISER 2026
- **Colonnes** : 14 (ID, OBJET, DESCRIPTION, STATUT, PRIORITE...)

In [ ]:
# ============================================================
# IMPORTS
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

print("✅ Imports OK")

---
## 1. Chargement des Données

In [ ]:
# ============================================================
# CHARGEMENT DU CSV
# ============================================================
df = pd.read_csv('../../data/Ticket_export.csv', encoding='utf-8',
                 sep=None, engine='python')

print(f"✅ Dataset chargé")
print(f"   Lignes    : {df.shape[0]}")
print(f"   Colonnes  : {df.shape[1]}")
print(f"\nColonnes : {df.columns.tolist()}")
df.head(3)

---
## 2. Exploration Générale
### Types de données et valeurs manquantes

In [ ]:
# ============================================================
# INFOS GÉNÉRALES + VALEURS MANQUANTES
# ============================================================
print("=== TYPES DE DONNÉES ===")
print(df.dtypes)

print("\n=== VALEURS MANQUANTES ===")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({
    'Manquants': missing,
    'Pourcentage (%)': missing_pct
}).sort_values('Pourcentage (%)', ascending=False)

print(missing_df[missing_df['Manquants'] > 0])

---
### Distribution des Statuts

In [ ]:
# ============================================================
# DISTRIBUTION STATUTS
# ============================================================
print("=== STATUTS ===")
print(df['STATUT'].value_counts())
print(f"\nValeurs uniques : {df['STATUT'].nunique()}")

plt.figure(figsize=(8, 4))
df['STATUT'].value_counts().plot(kind='bar', color='#534AB7')
plt.title('Distribution des statuts de tickets')
plt.xlabel('Statut')
plt.ylabel('Nombre de tickets')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

---
### Distribution des Priorités
> ⚠️ Observation : 99.5% des tickets ont la priorité "Normale"
> → La priorité sera générée automatiquement par règles métier

In [ ]:
# ============================================================
# DISTRIBUTION PRIORITÉS
# ============================================================
print("=== PRIORITÉS ===")
print(df['PRIORITE'].value_counts())

plt.figure(figsize=(6, 4))
df['PRIORITE'].value_counts().plot(kind='bar', color='#1D9E75')
plt.title('Distribution des priorités (données brutes)')
plt.xlabel('Priorité')
plt.ylabel('Nombre de tickets')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

---
### Top 10 Sites et Agents

In [ ]:
# ============================================================
# TOP 10 SITES
# ============================================================
plt.figure(figsize=(12, 4))
df['SITE'].value_counts().head(10).plot(kind='barh', color='#185FA5')
plt.title('Top 10 Sites')
plt.xlabel('Nombre de tickets')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

# TOP 10 AGENTS
plt.figure(figsize=(12, 4))
df['ATTRIBUÉ À'].value_counts().head(10).plot(kind='barh', color='#D85A30')
plt.title('Top 10 Agents')
plt.xlabel('Nombre de tickets')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

---
### Analyse du Texte (OBJET)
Distribution de la longueur des objets de tickets

In [ ]:
# ============================================================
# LONGUEUR DES OBJETS
# ============================================================
df['objet_length'] = df['OBJET'].str.len()

plt.figure(figsize=(10, 4))
df['objet_length'].hist(bins=50, color='#534AB7', edgecolor='white')
plt.title('Distribution longueur des objets de tickets')
plt.xlabel('Nombre de caractères')
plt.ylabel('Fréquence')
plt.tight_layout()
plt.show()

print(f"Longueur moyenne : {df['objet_length'].mean():.0f} caractères")
print(f"Longueur max     : {df['objet_length'].max()} caractères")
print(f"Longueur min     : {df['objet_length'].min()} caractères")

---
## 3. Nettoyage du Texte

### Problèmes identifiés
- Balises HTML : `&nbsp;`, `<br>`, `<p>`
- Caractères spéciaux et entités HTML
- Textes en majuscules
- Espaces multiples

### Solution
Fonction `clean_html()` appliquée sur OBJET et DESCRIPTION

In [ ]:
# ============================================================
# FONCTION NETTOYAGE HTML
# ============================================================
def clean_html(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub(r'<[^>]+>', ' ', text)        # balises HTML
    text = re.sub(r'&\w+;', ' ', text)           # entités HTML
    text = re.sub(r'http\S+', ' ', text)          # URLs
    text = re.sub(r'[^\w\sÀ-ÿ]', ' ', text)     # caractères spéciaux
    text = re.sub(r'\s+', ' ', text)              # espaces multiples
    return text.strip().lower()

# Appliquer sur OBJET et DESCRIPTION
df['OBJET_CLEAN'] = df['OBJET'].apply(clean_html)
df['DESC_CLEAN']  = df['DESCRIPTION'].apply(clean_html)

# ──────────────────────────────────────────────────────────────
# COLONNES GÉNÉRÉES :
#   - objet       → objet nettoyé seul (sujet du ticket)
#   - description → description nettoyée seule
#   - text        → objet + description combinés
#                   → c'est cette colonne qu'on utilise pour le ML (TF-IDF)
#                     car elle donne plus d'information au modèle
# ──────────────────────────────────────────────────────────────
df['TEXT_COMBINE'] = df['OBJET_CLEAN'] + ' ' + df['DESC_CLEAN']

print("✅ Nettoyage appliqué")
print(f"\nExemple AVANT : {df['DESCRIPTION'].iloc[1][:150]}")
print(f"\nExemple APRÈS : {df['DESC_CLEAN'].iloc[1][:150]}")

---
## 4. Génération Automatique des Catégories

### Stratégie
Comme le champ GROUPE est manquant à 99.63%, on génère 
les catégories automatiquement par **règles basées sur mots-clés**.

### Catégories définies (adaptées au contexte GEISER)
| Catégorie | Description |
|---|---|
| Réseau | Problèmes réseau, wifi, VPN, câblage |
| Matériel | PC, écran, imprimante, caméra, serveur |
| Logiciel | Sage, SwibTime, CarpRo, Ulysse, Office |
| Accès & Compte | Mots de passe, comptes, droits, 2FA |
| Email & Messagerie | Messagerie, Outlook, boîte mail |
| Téléphone | VoIP, extensions, appels |
| RH_Finance | Employés, paie, pointage |
| Réclamation | Plaintes, litiges, insatisfaction |
| Intervention | Demandes d'assistance, technicien |
| Autre | Non classifié |

In [ ]:
# ============================================================
# CATÉGORIES PAR MOTS-CLÉS
# ============================================================
categories = {
    'Réseau': [
        'réseau', 'network', 'wifi', 'internet', 'connexion',
        'vpn', 'ip', 'lan', 'switch', 'routeur', 'câble',
        'déconnexion', 'ping', 'firewall', 'fibre', 'cablage'
    ],
    'Matériel': [
        'ordinateur', 'pc', 'écran', 'clavier', 'souris',
        'imprimante', 'camera', 'caméra', 'scanner', 'disque',
        'batterie', 'laptop', 'portable', 'moniteur', 'usb',
        'panne', 'ecran', 'materiel', 'réparation',
        'serveur', 'server', 'backup', 'sauvegarde', 'stockage',
        'nas', 'vmware', 'nvr', 'tpe', 'caisse'
    ],
    'Logiciel': [
        'sage', 'swibtime', 'swib', 'carpro', 'ulysse',
        'ulyssetunisie', 'negosse', 'logiciel', 'application',
        'office', 'windows', 'installer', 'mise à jour',
        'licence', 'erp', 'teams', 'zoom', 'adobe', 'antivirus',
        'paie', 'virus', 'malware'
    ],
    'Accès & Compte': [
        'mot de passe', 'password', 'compte', 'login',
        'utilisateur', 'mdp', 'réinitialisation', 'bloquer',
        'verrouillé', 'permission', 'droits', 'session',
        '2fa', 'authentification', 'accès'
    ],
    'Email & Messagerie': [
        'email', 'mail', 'messagerie', 'boite', 'smtp',
        'exchange', 'envoyer', 'recevoir', 'courrier', 'spam',
        'outlook', 'boîte mail', 'extension mail'
    ],
    'Téléphone': [
        'téléphone', 'telephone', 'extension', 'appel',
        'voip', 'poste', 'ligne', 'hotline'
    ],
    'RH_Finance': [
        'employé', 'insertion', 'nouvel employé', 'recalcul',
        'salaire', 'rh', 'congé', 'pointage', 'swib time'
    ],
    'Réclamation': [
        'réclamation', 'reclamation', 'plainte', 'litige',
        'insatisfait', 'signalement'
    ],
    'Intervention': [
        'intervention', 'demande d intervention',
        'demande assistance', 'assistance', 'visite', 'technicien'
    ],
}

def assign_category(objet):
    if pd.isna(objet):
        return 'Autre'
    objet_lower = str(objet).lower()
    for cat, keywords in categories.items():
        if any(kw in objet_lower for kw in keywords):
            return cat
    return 'Autre'

df['CATEGORIE_AUTO'] = df['OBJET_CLEAN'].apply(assign_category)

print("=== CATÉGORIES AUTO ===")
print(df['CATEGORIE_AUTO'].value_counts())
print(f"\nNombre de catégories : {df['CATEGORIE_AUTO'].nunique()}")
print(f"Taux classification  : "
      f"{((df['CATEGORIE_AUTO'] != 'Autre').sum() / len(df) * 100):.1f}%")

plt.figure(figsize=(14, 5))
colors = ['#534AB7','#1D9E75','#185FA5','#D85A30',
          '#BA7517','#993556','#E24B4A','#888780',
          '#0F6E56','#633806']
df['CATEGORIE_AUTO'].value_counts().plot(
    kind='bar',
    color=colors[:len(df['CATEGORIE_AUTO'].unique())]
)
plt.title('Distribution des catégories (10 classes)')
plt.xlabel('Catégorie')
plt.ylabel('Nombre de tickets')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

---
## 5. Génération Automatique des Priorités

### Problème
99.5% des tickets ont la priorité "Normale" dans Geiser
→ Inutilisable pour entraîner un modèle ML

### Solution
Génération par **règles métier** basées sur mots-clés :

| Priorité | Mots-clés déclencheurs |
|---|---|
| P1_URGENT | urgent, critique, bloqué, panne totale, crash |
| P2_HIGH | problème, erreur, ne fonctionne pas, bug |
| P4_LOW | demande, information, installation, nouveau |
| P3_NORMAL | tout le reste |

In [ ]:
# ============================================================
# PRIORITÉS PAR RÈGLES MÉTIER
# ============================================================
def assign_priority(objet, description):
    text = str(objet) + " " + str(description)
    text = text.lower()

    p1_keywords = [
        'urgent', 'critique', 'panne totale', 'bloqué',
        'production arrêtée', 'inaccessible', 'impossible',
        'ne démarre pas', 'crash', 'data loss'
    ]
    p2_keywords = [
        'problème', 'erreur', 'ne fonctionne pas',
        'dysfonctionnement', 'bug', 'lent', 'instable',
        'ne s ouvre pas', 'échec', 'probleme'
    ]
    p4_keywords = [
        'demande', 'information', 'question',
        'renseignement', 'nouveau', 'création',
        'installation', 'mise à jour', 'insertion'
    ]

    if any(kw in text for kw in p1_keywords):
        return 'P1_URGENT'
    elif any(kw in text for kw in p2_keywords):
        return 'P2_HIGH'
    elif any(kw in text for kw in p4_keywords):
        return 'P4_LOW'
    else:
        return 'P3_NORMAL'

df['PRIORITE_AUTO'] = df.apply(
    lambda row: assign_priority(row['OBJET'], row['DESCRIPTION']),
    axis=1
)

print("=== DISTRIBUTION PRIORITÉS AUTO ===")
print(df['PRIORITE_AUTO'].value_counts())
print(f"\nPourcentages :")
print((df['PRIORITE_AUTO'].value_counts() / len(df) * 100).round(2))

plt.figure(figsize=(8, 4))
colors_prio = ['#E24B4A', '#EF9F27', '#378ADD', '#639922']
df['PRIORITE_AUTO'].value_counts().plot(kind='bar', color=colors_prio)
plt.title('Distribution des priorités générées automatiquement')
plt.xlabel('Priorité')
plt.ylabel('Nombre de tickets')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

---
## 6. Export du Dataset Nettoyé

Export du dataset final prêt pour l'entraînement ML.

### Colonnes exportées (12 colonnes)
| Colonne | Description |
|---|---|
| id | Identifiant unique ticket |
| objet | Objet nettoyé |
| description | Description nettoyée |
| text | Objet + Description combinés |
| categorie | Catégorie générée automatiquement (10 classes) |
| priorite | Priorité générée automatiquement (4 classes) |
| statut | Statut du ticket |
| site | Site du ticket |
| demandeur | Nom du demandeur |
| assignee | Agent assigné |
| date_creation | Date de création du ticket |
| date_echeance | Date d'échéance (SLA) |

In [ ]:
# ============================================================
# EXPORT DATASET FINAL
# ============================================================
df_ml = df[[
    'ID', 'OBJET_CLEAN', 'DESC_CLEAN', 'TEXT_COMBINE',
    'CATEGORIE_AUTO', 'PRIORITE_AUTO', 'STATUT', 'SITE',
    'DEMANDEUR', 'ATTRIBUÉ À', 'DATE DE CRÉATION', "DATE D'ECHÉANCE"
]].copy()

df_ml.columns = [
    'id', 'objet', 'description', 'text',
    'categorie', 'priorite', 'statut', 'site',
    'demandeur', 'assignee', 'date_creation', 'date_echeance'
]

df_ml = df_ml[df_ml['text'].str.len() > 10].reset_index(drop=True)

print(f"✅ Dataset final : {df_ml.shape[0]} tickets, {df_ml.shape[1]} colonnes")
print(f"\nColonnes : {df_ml.columns.tolist()}")
print(f"\nDistribution catégories :")
print(df_ml['categorie'].value_counts())
print(f"\nDistribution priorités :")
print(df_ml['priorite'].value_counts())
print(f"\nValeurs manquantes :")
print(df_ml.isnull().sum()[df_ml.isnull().sum() > 0])

df_ml.to_csv('../../data/dataset_clean.csv', index=False, encoding='utf-8')
print(f"\n✅ Sauvegardé : data/dataset_clean.csv")

---
## 7. Labellisation des Sous-Catégories via LLM (Groq)

### Pourquoi ?
Le dataset ne contient pas de sous-catégories. On les génère automatiquement
en envoyant chaque ticket au modèle **LLaMA 3 (Groq API)** avec la catégorie
déjà connue comme contexte.

### Taxonomie des sous-catégories
| Catégorie | Sous-catégories |
|---|---|
| Email & Messagerie | Création compte email · Suppression compte email · Extension boîte mail · Problème accès email · Configuration email |
| Accès & Compte | Réinitialisation mot de passe · Création de compte · Désactivation compte · Authentification 2FA · Gestion des permissions |
| Matériel | PC & Ordinateur · Écran & Périphériques · Imprimante & Scanner · Caméra & Surveillance · Serveur & Infrastructure · TPE & Caisse · Autre matériel |
| Réseau | Connexion Internet · WiFi · Câblage réseau · Lenteur réseau · VPN |
| Logiciel | Installation logiciel · Mise à jour logiciel · Bug logiciel · Accès logiciel · Configuration logiciel |
| Téléphone | Téléphone fixe & VOIP · Téléphone mobile · Configuration téléphone |
| Réclamation | Réclamation service · Réclamation matériel · Réclamation logiciel |
| Intervention | Intervention sur site · Maintenance préventive · Déplacement technique |
| RH_Finance | Gestion RH · Gestion Finance & Paie · Autre RH/Finance |
| Autre | Demande générale · Non classifiable |

### Approche
- Modèle : `llama-3.1-8b-instant` (Groq — gratuit, 500K tokens/jour)
- Checkpoint JSON → reprise possible si interruption
- Produit : `dataset_labeled.csv`

In [ ]:
# ============================================================
# CONFIG LABELLISATION LLM
# ============================================================
import os, json, time
from pathlib import Path
from groq import Groq

# Charger la clé depuis backend/.env
env_path = Path('..') / '.env'
if env_path.exists():
    for line in env_path.read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if line and not line.startswith('#') and '=' in line:
            k, v = line.split('=', 1)
            os.environ.setdefault(k.strip(), v.strip())

GROQ_API_KEY  = os.environ.get('GROQ_API_KEY', '')
GROQ_MODEL    = 'llama-3.1-8b-instant'
CHECKPOINT    = '../../scripts/checkpoint.json'
DELAY_SECONDS = 0.3

TAXONOMY = {
    'Email & Messagerie': ['Création compte email', 'Suppression compte email',
                           'Extension boîte mail', 'Problème accès email', 'Configuration email'],
    'Accès & Compte':     ['Réinitialisation mot de passe', 'Création de compte',
                           'Désactivation compte', 'Authentification 2FA', 'Gestion des permissions'],
    'Matériel':           ['PC & Ordinateur', 'Écran & Périphériques', 'Imprimante & Scanner',
                           'Caméra & Surveillance', 'Serveur & Infrastructure', 'TPE & Caisse', 'Autre matériel'],
    'Réseau':             ['Connexion Internet', 'WiFi', 'Câblage réseau', 'Lenteur réseau', 'VPN'],
    'Logiciel':           ['Installation logiciel', 'Mise à jour logiciel', 'Bug logiciel',
                           'Accès logiciel', 'Configuration logiciel'],
    'Téléphone':          ['Téléphone fixe & VOIP', 'Téléphone mobile', 'Configuration téléphone'],
    'Réclamation':        ['Réclamation service', 'Réclamation matériel', 'Réclamation logiciel'],
    'Intervention':       ['Intervention sur site', 'Maintenance préventive', 'Déplacement technique'],
    'RH_Finance':         ['Gestion RH', 'Gestion Finance & Paie', 'Autre RH/Finance'],
    'Autre':              ['Demande générale', 'Non classifiable'],
}

def build_prompt(objet, description, categorie):
    sous_cats = TAXONOMY.get(categorie, ['Non classifiable'])
    liste = '|'.join(sous_cats)
    desc_courte = str(description)[:150] if description else ''
    return f"Catégorie: {categorie}\nTicket: {objet}. {desc_courte}\nChoix: {liste}\nRéponds uniquement avec un choix exact."

def load_checkpoint():
    if os.path.exists(CHECKPOINT):
        with open(CHECKPOINT, encoding='utf-8') as f:
            return json.load(f)
    return {}

def save_checkpoint(data):
    os.makedirs(os.path.dirname(CHECKPOINT), exist_ok=True)
    with open(CHECKPOINT, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

if not GROQ_API_KEY:
    print("⚠️  GROQ_API_KEY non trouvée dans .env — labellisation ignorée")
else:
    print(f"✅ Groq configuré : {GROQ_MODEL}")
    print(f"   Checkpoint     : {CHECKPOINT}")

In [ ]:
# ============================================================
# LABELLISATION SOUS-CATÉGORIES VIA GROQ
# ============================================================
if not GROQ_API_KEY:
    print("⚠️  Clé Groq manquante — sous_categorie = 'Non classifiable' pour tous")
    df_ml['sous_categorie'] = df_ml.apply(
        lambda r: TAXONOMY.get(r['categorie'], ['Non classifiable'])[0], axis=1
    )
else:
    client     = Groq(api_key=GROQ_API_KEY)
    checkpoint = load_checkpoint()
    total      = len(df_ml)
    errors     = 0

    print(f"Total tickets       : {total}")
    print(f"Déjà traités        : {len(checkpoint)}")
    print(f"Restants            : {total - len(checkpoint)}\n")

    sous_cats_list = []

    for i, row in df_ml.iterrows():
        ticket_id = str(row['id'])

        if ticket_id in checkpoint:
            sous_cats_list.append(checkpoint[ticket_id])
            continue

        try:
            response = client.chat.completions.create(
                model=GROQ_MODEL,
                messages=[
                    {'role': 'system',
                     'content': 'Tu es un classificateur de tickets IT. Réponds UNIQUEMENT avec le nom exact d\'une sous-catégorie, sans autre texte.'},
                    {'role': 'user',
                     'content': build_prompt(row['objet'], row['description'], row['categorie'])},
                ],
                temperature=0.0,
                max_tokens=30,
            )
            sous_cat = response.choices[0].message.content.strip()
            valid = TAXONOMY.get(row['categorie'], [])
            if sous_cat not in valid:
                sous_cat = valid[0] if valid else 'Non classifiable'

            sous_cats_list.append(sous_cat)
            checkpoint[ticket_id] = sous_cat

            if (i + 1) % 50 == 0:
                save_checkpoint(checkpoint)

            if (i + 1) % 200 == 0:
                print(f"  [{i+1}/{total}] {row['categorie']} → {sous_cat}")

            time.sleep(DELAY_SECONDS)

        except Exception as e:
            valid = TAXONOMY.get(row['categorie'], ['Non classifiable'])
            sous_cat = valid[0]
            sous_cats_list.append(sous_cat)
            errors += 1
            time.sleep(2)

    save_checkpoint(checkpoint)
    df_ml['sous_categorie'] = sous_cats_list
    print(f"\n✅ Labellisation terminée — Erreurs : {errors}")

print(f"\nDistribution sous-catégories (top 10) :")
print(df_ml['sous_categorie'].value_counts().head(10))

In [ ]:
# ============================================================
# EXPORT DATASET LABELED FINAL
# ============================================================
df_ml.to_csv('../../data/dataset_labeled.csv', index=False, encoding='utf-8')

print(f"✅ Sauvegardé : data/dataset_labeled.csv")
print(f"   Lignes   : {df_ml.shape[0]}")
print(f"   Colonnes : {df_ml.shape[1]} → {df_ml.columns.tolist()}")
print(f"\nAperçu :")
df_ml[['id', 'objet', 'categorie', 'sous_categorie', 'priorite']].head(5)

---
## 8. Résumé

| Étape | Résultat |
|---|---|
| Dataset brut | 1915 tickets, 14 colonnes |
| Colonnes ignorées | GROUPE (99.6% vide), EST_DEMANDE_SERVICE, TEMPLATE_NOM, TEMPLATE_SERVICE |
| Colonnes conservées | 12 colonnes utiles |
| Nettoyage HTML | OBJET + DESCRIPTION nettoyés |
| Catégories générées | 10 classes par règles mots-clés |
| Priorités générées | 4 classes (P1→P4) par règles métier |
| Sous-catégories | Via LLM Groq (llama-3.1-8b-instant) |
| dataset_clean.csv | 1915 tickets, 12 colonnes, sans sous-catégorie |
| dataset_labeled.csv | 1915 tickets, 13 colonnes, avec sous-catégorie |

### Prochaine étape
**Notebook 02** — Entraînement des modèles ML (catégorie + sous-catégorie + priorité)